<a href="https://colab.research.google.com/github/jonash-chataut/GenAI/blob/main/tool_calling_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import userdata
HUGGINGFACEHUB_ACCESS_TOKEN=userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

In [2]:
!pip install -U langchain langchain-community langchain-core langchainhub langchain-openai
!pip install -q langchain_huggingface
!pip install -q langchain-core requests

In [5]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm_hf=HuggingFaceEndpoint(
    repo_id="MiniMaxAI/MiniMax-M2.5",
    task="text-generation",
    huggingfacehub_api_token=HUGGINGFACEHUB_ACCESS_TOKEN
)
llm=ChatHuggingFace(llm=llm_hf)

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [7]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [8]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [9]:
multiply.name

'multiply'

In [10]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [11]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [12]:
# tool binding

In [13]:
llm.invoke('hi')

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 38, 'total_tokens': 80}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d0915-b91f-7742-b826-651feefb2770-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 42, 'total_tokens': 80})

In [14]:
llm_with_tools = llm.bind_tools([multiply])

In [15]:
llm_with_tools.invoke('Hi how are you')


AIMessage(content="Hi there! I'm doing well, thank you for asking! I'm here and ready to help you with whatever you need. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 207, 'total_tokens': 274}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d0915-e43b-7a11-b4e9-c9ad948a31e5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 207, 'output_tokens': 67, 'total_tokens': 274})

In [16]:
query = HumanMessage('can you multiply 3 with 1000')

In [17]:
messages = [query]

In [18]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [19]:
result = llm_with_tools.invoke(messages)

In [20]:
messages.append(result)

In [21]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'chatcmpl-tool-832ee6e6930d034e', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 212, 'total_tokens': 314}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0916-5201-7241-9769-fa5df5e9b1bb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'chatcmpl-tool-832ee6e6930d034e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 212, 'output_tokens': 102, 'total_tokens': 314})]

In [22]:
tool_result = multiply.invoke(result.tool_calls[0])

In [23]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='chatcmpl-tool-832ee6e6930d034e')

In [24]:
messages.append(tool_result)

In [25]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'chatcmpl-tool-832ee6e6930d034e', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 212, 'total_tokens': 314}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0916-5201-7241-9769-fa5df5e9b1bb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'chatcmpl-tool-832ee6e6930d034e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 212, 'output_tokens': 102, 'total_tokens': 314}),
 ToolMessage(content='3000', name='multiply', tool_call_id='chatcmpl-tool-832ee6e6930d034e')]

In [26]:
llm_with_tools.invoke(messages).content

'3 multiplied by 1000 equals **3000**.'

In [48]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/04aad474c627dd4ae6bca6a5/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  This tool converts a currency value using a conversion rate obtained from another tool.
  The conversion_rate is automatically provided from previous tool output.
  """

  return base_currency_value * conversion_rate

# Create tools list
tools = [get_conversion_factor, convert]


In [28]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [29]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'NPR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1773964801,
 'time_last_update_utc': 'Fri, 20 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1774051201,
 'time_next_update_utc': 'Sat, 21 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'NPR',
 'conversion_rate': 148.9119}

In [30]:
convert.invoke({'base_currency_value':10, 'conversion_rate':147.9598})

1479.598

In [31]:
# tool binding
llm=ChatHuggingFace(llm=llm_hf)

In [32]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [33]:
messages = [HumanMessage('What is the conversion factor between USD and NPR, and based on that convert 100 usd to npr')]

In [34]:
messages

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that convert 100 usd to npr', additional_kwargs={}, response_metadata={})]

In [35]:
ai_message = llm_with_tools.invoke(messages)

In [36]:
messages.append(ai_message)

In [37]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'chatcmpl-tool-ab4a2f779e49dcac',
  'type': 'tool_call'}]

In [38]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [39]:
messages

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that convert 100 usd to npr', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'll help you find the conversion factor between USD and NPR, and then convert 100 USD to NPR. Let me first get the conversion factor.", additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "NPR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'chatcmpl-tool-ab4a2f779e49dcac', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 311, 'total_tokens': 462}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0918-8d25-7953-8690-925a02b88290-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'chatcmpl-tool-ab4a2f779e49dcac', 'type': 'tool_call'}], inv

In [40]:
llm_with_tools.invoke(messages).content

''

In [52]:
!pip install langgraph

In [54]:
from langgraph.prebuilt import create_react_agent

# Create the agent using LangGraph
agent_executor = create_react_agent(llm, tools)

/tmp/ipykernel_782/720670605.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


In [58]:
# Example usage
result = agent_executor.invoke({
    "messages": [HumanMessage(content="Convert 100 USD to NPR")]
})
# print(result)
final_message = result["messages"][-1]
print(final_message.content)

100 USD is equal to **14,891.19 NPR**.

This conversion uses the exchange rate of 1 USD = 148.9119 NPR.
